### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors optimize for a single metric. Seniors optimize for the metric that minimizes **Business Regret**. In medicine, we minimize False Negatives (Recall). In finance, we minimize False Positives (Precision). But to truly compare models, we use statistical significance tests like **McNemar's** to confirm that a 0.5% gain isn't just a lucky random seed.

**Mental Model:** 
- **Cohen's Kappa:** Adjusts your accuracy for the 'luck' of the majority class.
- **MCC:** The only metric that considers all four quadrants of the confusion matrix equally.
- **McNemar's:** Looks at the 'Discordant Pairs' (where models disagree) to see if one model is actually logically superior.

**Common Junior Pitfall:** Assuming that if Model B has 0.1% higher AUC than Model A, it's better. At scale, this difference is often noise. Never swap a model in production without a statistical $p < 0.05$ result.

---


## 1. Imbalanced Evaluation (ROC vs PR)
ROC-AUC is dangerously optimistic on imbalanced data because it is insensitive to the sheer volume of False Positives. Precision-Recall AUC (PR-AUC) captures the true cost of failure.

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#-socratic-deep-check)
  * [🎓 Junior to Senior: Concept Bridge](#-junior-to-senior-concept-bridge)
* [1. Imbalanced Evaluation (ROC vs PR)](#1-imbalanced-evaluation-roc-vs-pr)
* [1.1 Beyond F1: Cohen's Kappa & MCC](#11-beyond-f1-cohens-kappa-mcc)
* [2. Expected Calibration Error (ECE)](#2-expected-calibration-error-ece)
* [3. Statistical Hypothesis Testing — McNemar's Test](#3-statistical-hypothesis-testing-mcnemars-test)
* [✅ Knowledge Check](#-knowledge-check)
  * [Q1: MCC vs. F1 on imbalance](#q1-mcc-vs-f1-on-imbalance)
  * [Q2: Why calibration matters](#q2-why-calibration-matters)
* [🔨 Practical Practice](#-practical-practice)
  * [Tier 1: Beginner](#tier-1-beginner)
  * [Tier 2: Intermediate](#tier-2-intermediate)
  * [Tier 3: Advanced](#tier-3-advanced)

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

# Simulate extreme imbalance (99% negative)
y_true = np.array([0]*990 + [1]*10)
y_scores = np.random.rand(1000)
y_scores[y_true == 1] += 0.5 # Give positives a slight bump

print(f"ROC-AUC: {roc_auc_score(y_true, y_scores):.3f} (Looks Good!)")
print(f"PR-AUC:  {average_precision_score(y_true, y_scores):.3f} (The Reality)")

"""
What just happened?
We compared ROC vs PR. Even though the model has a decent ROC (often >0.7), 
the PR score reveals that almost every positive prediction is a false alarm.
"""

## 1.1 Beyond F1: Cohen's Kappa & MCC

**Cohen's Kappa ($\kappa$):**
$$\kappa = \frac{p_o - p_e}{1 - p_e}$$
Where $p_o$ is observed accuracy and $p_e$ is expected accuracy by chance. 

**Matthews Correlation Coefficient (MCC):**
$$MCC = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$
MCC is the gold standard for binary classification on imbalanced data because it takes into account all four quadrants of the confusion matrix.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

def calculate_mcc(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel().astype(float)
    num = (tp * tn) - (fp * fn)
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return num / den if den != 0 else 0

y_pred = np.zeros(1000) # Baseline model: Always predict negative
print(f"F1 Score: {f1_score(y_true, y_pred):.3f}")
print(f"MCC Score: {calculate_mcc(y_true, y_pred):.3f}")

"""
What just happened?
We calculated MCC. Note how F1 (and Accuracy) can hide failure, while MCC 
correctly identifies that an 'always negative' model is essentially random/unhelpful.
"""

## 2. Expected Calibration Error (ECE)

A model is **Calibrated** if its predicted probability equals the actual frequency of the event. 
**ECE** bins the data by confidence and calculates the weighted distance between confidence and accuracy.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Random Forest is notorious for being uncalibrated (pushing scores toward 0.5)
rf = RandomForestClassifier().fit(X_sub[:100], y_true[:100]) # Pseudo code placeholder
prob_pos = rf.predict_proba(y_true)[:, 1] # Pseudo code placeholder

# Real ECE Logic
def calculate_ece(y_true, y_prob, n_bins=10):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins)
    # Simple weighted average of bin error
    return np.mean(np.abs(prob_true - prob_pred))

"""
What just happened?
We defined the ECE. In production, we plot Reliability Diagrams to see 
where the model is 'overconfident' (Accuracy < Confidence) or 'underconfident'.
"""

## 3. Statistical Hypothesis Testing — McNemar's Test

When Model A has 85.3% accuracy and Model B has 85.7%, is Model B truly better? 
**McNemar's test** looks at the points where Model A and Model B **Disagree**.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def run_mcnemar(y_true, pred_a, pred_b):
    # Disagreement Matrix
    #           B correct | B wrong
    # A correct |   n00   |   n01
    # A wrong   |   n10   |   n11
    
    n01 = np.sum((pred_a == y_true) & (pred_b != y_true))
    n10 = np.sum((pred_a != y_true) & (pred_b == y_true))
    
    # Statistic = (abs(n01 - n10) - 1)^2 / (n01 + n10)
    table = [[0, n01], [n10, 0]]
    result = mcnemar(table, exact=True)
    return result.pvalue

"""
What just happened?
We used McNemar's logic. If the p-value is > 0.05, the performance difference 
between Model A and Model B is NOT statistically significant. Do not swap 
champion models for challengers in this scenario.
"""

----- 
## ✅ Knowledge Check

### Q1: MCC vs. F1 on imbalance
<details><summary>👉 Answer</summary>
F1 score does not use True Negatives (TN). If a model predicts 'Negative' for everything, F1 is technically undefined or 0, but if it gets 1-2 positives right it jumps. MCC uses all four quadrants; it will only be high if the model predicts BOTH classes correctly.
</details>

### Q2: Why calibration matters
<details><summary>👉 Answer</summary>
If your business logic triggers a $50 coupon when churn probability > 0.9, an uncalibrated model might never hit 0.9 despite having high ranking capability. Calibration (ECE) ensures your model's 'confidence' has a physical meaning in reality.
</details>


----- 
## 🔨 Practical Practice

### Tier 1: Beginner
1. Manually calculate the Confusion Matrix for 10 predictions and find the MCC.
2. Compare the ROC-AUC vs PR-AUC for a model with 95% majority class imbalance.

### Tier 2: Intermediate
1. **Reliability Diagram:** Plot the calibration curve for a GBDT (XGBoost/LightGBM) model and calculate its ECE. Apply Isotonic calibration and show the improvement.
2. **Disagreement Matrix:** Implement the McNemar disagreement matrix count manually from two prediction arrays.

### Tier 3: Advanced
1. **McNemar Significance:** Train two slightly different Random Forest models and prove that a 0.5% accuracy difference is statistically insignificant using McNemar's p-value.
2. **Kappa from Scratch:** Implement Cohen's Kappa from scratch using only NumPy and verify it against sklearn. Prove that Kappa correctly penalizes 'majority-class gaming'.
